# Lesson 11 - Model Context Protocol (MCP)

The **Model Context Protocol (MCP)** is an open standard that enables agents to dynamically discover and use tools, resources, and data sources at runtime. Instead of hardcoding tools into an agent, MCP lets agents connect to external servers that expose capabilities on demand.

In this lesson, you'll learn:
- What MCP is and why it matters for agent systems
- How the MCP client-server architecture works
- How to build agents that use MCP-style tool discovery


## Ρύθμιση

**Απαιτούμενα:**
- Έργο Microsoft Foundry με αναπτυγμένο μοντέλο
- Εκτελέστε `az login` για αυθεντικοποίηση


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import dotenv
from typing import Annotated
from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## Τι είναι το Πρωτόκολλο Πλαισίου Μοντέλου (MCP);

Το MCP ορίζει έναν πρότυπο τρόπο για τους πράκτορες AI να εντοπίζουν και να αλληλεπιδρούν με εξωτερικά εργαλεία και πηγές δεδομένων:

- **MCP Server**: Εκθέτει εργαλεία, πόρους και προτροπές μέσω ενός τυπικού πρωτοκόλλου
- **MCP Client**: Το περιβάλλον χρόνου εκτέλεσης πράκτορα που συνδέεται σε διακομιστές και ανακαλύπτει διαθέσιμες δυνατότητες
- **Dynamic Discovery**: Οι πράκτορες δεν χρειάζονται ενσωματωμένα εργαλεία — ανακαλύπτουν τι είναι διαθέσιμο κατά το χρόνο εκτέλεσης

Αυτό είναι ισχυρό για την κατασκευή επεκτάσιμων συστημάτων πρακτόρων όπου νέες δυνατότητες μπορούν να προστεθούν χωρίς να τροποποιηθεί ο κώδικας του πράκτορα.


## Πώς Λειτουργεί το MCP

```
┌─────────────┐     discover      ┌─────────────────┐
│  MCP Client  │ ──────────────► │   MCP Server     │
│  (Agent)     │                  │  (Tool Provider) │
│              │ ◄────────────── │                   │
│              │   tool results   │  • list_tools()  │
│              │                  │  • call_tool()   │
└─────────────┘                  │  • resources     │
                                  └─────────────────┘
```

1. Ο πράκτορας (πελάτης MCP) συνδέεται σε έναν διακομιστή MCP
2. Ο διακομιστής ανταποκρίνεται με μια λίστα διαθέσιμων εργαλείων και τα σχήματά τους
3. Ο πράκτορας μπορεί στη συνέχεια να καλέσει οποιοδήποτε ανακαλυφθέν εργαλείο κατά τη διαδικασία συλλογισμού του
4. Τα αποτελέσματα επιστρέφουν μέσω του ίδιου πρωτοκόλλου


## Προσομοίωση Ανίχνευσης Εργαλείων MCP

Δεδομένου ότι ένας πραγματικός διακομιστής MCP απαιτεί μια τρέχουσα διαδικασία διακομιστή, θα παρουσιάσουμε το πρότυπο χρησιμοποιώντας συναρτήσεις `@tool` που προσομοιώνουν αυτό που θα παρείχε μια υπηρεσία καταλύματος συνδεδεμένη σε MCP.

Στην παραγωγή, αυτά τα εργαλεία θα ανακαλύπτονται δυναμικά από έναν διακομιστή MCP αντί να ορίζονται τοπικά.


In [ ]:
@tool(approval_mode="never_require")
def search_accommodations(
    location: Annotated[str, "The city to search for accommodations"],
    check_in: Annotated[str, "Check-in date (YYYY-MM-DD)"],
    check_out: Annotated[str, "Check-out date (YYYY-MM-DD)"],
    guests: Annotated[int, "Number of guests"] = 2
) -> str:
    """Search for accommodations (simulating an MCP-connected Airbnb tool).
    In production, this would be discovered via MCP from an accommodation service."""
    listings = {
        "Tokyo": [
            {"name": "Shinjuku Modern Apartment", "price": 120, "rating": 4.8},
            {"name": "Traditional Ryokan in Asakusa", "price": 200, "rating": 4.9},
            {"name": "Shibuya Studio", "price": 85, "rating": 4.5},
        ],
        "Paris": [
            {"name": "Le Marais Charming Flat", "price": 150, "rating": 4.7},
            {"name": "Montmartre Artist Loft", "price": 110, "rating": 4.6},
        ],
        "Barcelona": [
            {"name": "Gothic Quarter Penthouse", "price": 130, "rating": 4.8},
            {"name": "Barceloneta Beach Flat", "price": 95, "rating": 4.4},
        ],
    }
    results = listings.get(location, [])
    if not results:
        return f"No accommodations found in {location}"
    output = f"Accommodations in {location} ({check_in} to {check_out}, {guests} guests):\n"
    for listing in results:
        output += f"  - {listing['name']}: ${listing['price']}/night (★{listing['rating']})\n"
    return output


@tool(approval_mode="never_require")
def get_local_experiences(
    location: Annotated[str, "The city to find experiences in"],
    interest: Annotated[str, "Type of experience (food, culture, adventure, etc.)"] = "all"
) -> str:
    """Get local experiences and activities (simulating an MCP-connected tourism tool)."""
    experiences = {
        "Tokyo": {
            "food": ["Tsukiji Market Tour ($45)", "Ramen Making Class ($60)", "Sake Tasting ($35)"],
            "culture": ["Tea Ceremony ($50)", "Samurai Museum ($15)", "Sumo Tournament ($80)"],
            "adventure": ["Mt. Fuji Day Trip ($120)", "Go-kart City Tour ($80)"],
        },
        "Paris": {
            "food": ["Wine & Cheese Tasting ($55)", "Cooking Class ($90)", "Market Tour ($40)"],
            "culture": ["Louvre Guided Tour ($35)", "Montmartre Art Walk ($25)"],
        },
    }
    city_exp = experiences.get(location, {})
    if not city_exp:
        return f"No experiences found in {location}"
    if interest != "all" and interest in city_exp:
        items = city_exp[interest]
        return f"{interest.title()} experiences in {location}:\n" + "\n".join(f"  - {e}" for e in items)
    output = f"All experiences in {location}:\n"
    for cat, items in city_exp.items():
        output += f"\n  {cat.title()}:\n"
        for item in items:
            output += f"    - {item}\n"
    return output

## Δημιουργία ενός Αντιπροσώπου με Εργαλεία Στυλ MCP


In [ ]:
agent = client.as_agent(
    tools=[search_accommodations, get_local_experiences],
    name="AccommodationAgent",
    instructions="""You are an accommodation and travel experiences specialist powered by MCP-connected services.

Help travelers find the perfect place to stay and things to do. When searching:
1. Use the search_accommodations tool to find listings
2. Use the get_local_experiences tool to suggest activities
3. Compare options and make personalized recommendations
4. Consider the traveler's budget, interests, and travel style""",
)

response = await agent.run(
    "I'm visiting Tokyo for 5 nights in April with my partner. We love traditional Japanese culture and food. "
    "Find us a place to stay and suggest some experiences.",
    )
print(response)

## MCP σε Παραγωγή

Σε ένα περιβάλλον παραγωγής, το MCP επιτρέπει ισχυρά πρότυπα:

- **Δυναμική ανακάλυψη εργαλείων**: Οι πράκτορες συνδέονται με τους διακομιστές MCP και ανακαλύπτουν εργαλεία κατά την εκτέλεση
- **Αποσυνδεδεμένη αρχιτεκτονική**: Οι πάροχοι εργαλείων μπορούν να ενημερώνονται ανεξάρτητα από τον πράκτορα
- **Διαμοιρασμός μεταξύ οργανισμών**: Οι ομάδες μπορούν να εκθέτουν δυνατότητες μέσω MCP servers που κάθε πράκτορας μπορεί να χρησιμοποιήσει
- **Υποστήριξη Microsoft Agent Framework**: Το MAF έχει ενσωματωμένη υποστήριξη πελάτη MCP μέσω της ενσωμάτωσης `mcp`

Για να χρησιμοποιήσετε έναν πραγματικό διακομιστή MCP με το MAF, θα συνδεθείτε μέσω του `hosted_mcp_tool()` ή της ενσωμάτωσης πελάτη MCP.

**Μάθετε περισσότερα:**
- [MCP Specification](https://modelcontextprotocol.io/)
- [Microsoft Agent Framework MCP Support](https://github.com/microsoft/agent-framework/tree/main/python/samples/02-agents/mcp)


## Σύνοψη

Σε αυτό το μάθημα, μάθατε:
- Το **MCP** είναι ένα ανοιχτό πρότυπο για δυναμική ανακάλυψη εργαλείων μεταξύ πρακτόρων και παρόχων εργαλείων
- Η **αρχιτεκτονική πελάτη-εξυπηρετητή** επιτρέπει στους πράκτορες να ανακαλύπτουν δυνατότητες κατά τη διάρκεια εκτέλεσης
- Το MCP επιτρέπει **επεκτάσιμα, αποσυνδεδεμένα συστήματα πρακτόρων** όπου μπορούν να προστεθούν εργαλεία χωρίς αλλαγές στον κώδικα
- Το Microsoft Agent Framework παρέχει **ενσωματωμένη υποστήριξη MCP** για παραγωγική χρήση


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Αποποίηση ευθυνών**:
Αυτό το έγγραφο έχει μεταφραστεί χρησιμοποιώντας την υπηρεσία μετάφρασης με τεχνητή νοημοσύνη [Co-op Translator](https://github.com/Azure/co-op-translator). Ενώ επιδιώκουμε την ακρίβεια, παρακαλούμε να έχετε υπόψη ότι οι αυτοματοποιημένες μεταφράσεις ενδέχεται να περιέχουν λάθη ή ανακρίβειες. Το πρωτότυπο έγγραφο στη μητρική του γλώσσα πρέπει να θεωρείται η αυθεντική πηγή. Για κρίσιμες πληροφορίες, συνιστάται επαγγελματική ανθρώπινη μετάφραση. Δεν φέρουμε ευθύνη για τυχόν παρεξηγήσεις ή λανθασμένες ερμηνείες που προκύπτουν από τη χρήση αυτής της μετάφρασης.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
